# 01 — Baseline evaluation

Frozen-policy baselines on the cybersec OpenEnv. Runs:

* `RandomPolicy`
* `HeuristicPolicy`
* (Optional) `Qwen2.5-1.5B-Instruct` zero-shot defender via JSON-only prompts.

for 50 seeds × 3 scenarios and dumps:

* `_artifacts/baseline_metrics.json` — for the post-training notebook to diff against
* `_artifacts/baseline_curves.png` — reward curves per policy
* `_artifacts/trajectory_dataset.jsonl` — heuristic trajectories for optional SFT warm-up

Designed to fit a Colab T4 / Kaggle P100 in ~10 minutes (without the LLM cell).

## 0. Setup

In [ ]:
%%capture
# Run this cell on a fresh Colab/Kaggle kernel.
# The env package lives in ../cybersec/ per the openenv init layout.
!pip install -q -e ../cybersec[notebooks]

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from cybersec import CybersecEnvironment, list_scenarios
from cybersec.baselines import (
    HeuristicPolicy,
    RandomPolicy,
    aggregate_results,
    run_episode,
)

ARTIFACTS = Path("_artifacts")
ARTIFACTS.mkdir(exist_ok=True)
N_EPISODES = 50
SCENARIOS = list_scenarios()
print("scenarios:", SCENARIOS)

## 1. Random vs Heuristic — 50 episodes per scenario

In [ ]:
env = CybersecEnvironment()
seeds = list(range(N_EPISODES))
results = {}

for scenario_id in SCENARIOS:
    rand = [run_episode(env, RandomPolicy(seed=s), seed=s, scenario_id=scenario_id) for s in seeds]
    heur = [run_episode(env, HeuristicPolicy(),        seed=s, scenario_id=scenario_id) for s in seeds]
    results[scenario_id] = {"random": rand, "heuristic": heur}
    print(scenario_id)
    print("  random   :", aggregate_results(rand))
    print("  heuristic:", aggregate_results(heur))

## 2. Reward-curve plot (mean ± std across seeds)

In [ ]:
def _padded_curve(curves, target_len):
    out = np.zeros((len(curves), target_len), dtype=float)
    for i, c in enumerate(curves):
        out[i, : len(c)] = c
        if len(c) < target_len:
            out[i, len(c):] = c[-1] if c else 0.0
    return np.cumsum(out, axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, scenario_id in zip(axes, SCENARIOS):
    cell = results[scenario_id]
    horizon = max(max(len(r.reward_curve) for r in cell['random']),
                  max(len(r.reward_curve) for r in cell['heuristic']))
    for label, color in [("random", "tab:gray"), ("heuristic", "tab:blue")]:
        cumr = _padded_curve([r.reward_curve for r in cell[label]], horizon)
        mean = cumr.mean(axis=0)
        std = cumr.std(axis=0)
        ax.plot(mean, label=label, color=color)
        ax.fill_between(np.arange(horizon), mean - std, mean + std, color=color, alpha=0.15)
    ax.set_title(scenario_id, fontsize=10)
    ax.set_xlabel("tick")
    ax.axhline(0, color="k", lw=0.5)
axes[0].set_ylabel("cumulative reward")
axes[0].legend()
fig.suptitle(f"Cybersec OpenEnv — baselines, n={N_EPISODES}/scenario")
fig.tight_layout()
fig.savefig(ARTIFACTS / "baseline_curves.png", dpi=140)
plt.show()

## 3. Persist metrics for the post-training notebook

In [ ]:
metrics = {
    scenario_id: {
        policy: aggregate_results(cell[policy]) for policy in ("random", "heuristic")
    }
    for scenario_id, cell in results.items()
}
metrics["_meta"] = {"n_episodes": N_EPISODES, "seeds": list(range(N_EPISODES)), "scenarios": SCENARIOS}
(ARTIFACTS / "baseline_metrics.json").write_text(json.dumps(metrics, indent=2))
print("wrote", ARTIFACTS / "baseline_metrics.json")

## 4. Heuristic trajectory dataset (for optional SFT warm-up)

We dump observation/action pairs into JSONL so notebook 02 can do a few hundred steps of SFT before GRPO. Each line:

```json
{"prompt": "<rendered observation>", "completion": "<json action>", "reward": <step reward>}
```

In [ ]:
def render_observation(obs) -> str:
    """Compact prompt format reused by notebooks 02/03."""
    lines = [
        f"tick={obs.tick}/{obs.horizon}  scenario={obs.scenario_id}  attacker={obs.attacker_personality.value}",
        f"isolated_assets={obs.isolated_assets}",
        f"revoked_identities={obs.revoked_identities}",
        f"blocked_egress={obs.blocked_egress_assets}",
        f"patched={obs.patched_assets}",
        f"confirmed_compromised={obs.confirmed_compromised}",
        f"valid_targets={obs.valid_targets}",
        "recent_alerts:",
    ]
    for a in obs.alerts[-6:]:
        lines.append(f"  t={a.tick} {a.signal.value} sev={a.severity} asset={a.asset} id={a.identity} :: {a.description}")
    lines.append("recent_forensics:")
    for f in obs.forensics[-4:]:
        lines.append(f"  t={f.tick} {f.target_kind}={f.target} compromised={f.is_compromised} conf={f.confidence}")
    return "\n".join(lines)

out_path = ARTIFACTS / "trajectory_dataset.jsonl"
n_pairs = 0
with open(out_path, "w", encoding="utf-8") as fh:
    for scenario_id in SCENARIOS:
        for s in seeds:
            policy = HeuristicPolicy()
            obs = env.reset(seed=s, scenario_id=scenario_id)
            while not obs.done:
                act = policy.act(obs)
                prompt = render_observation(obs)
                completion = act.model_dump_json()
                obs = env.step(act)
                step_reward = (obs.info or {}).get("reward_breakdown", {}).get("total", 0.0)
                fh.write(json.dumps({"prompt": prompt, "completion": completion, "reward": step_reward}) + "\n")
                n_pairs += 1
print(f"wrote {n_pairs} (prompt, completion) pairs to {out_path}")

## 5. (Optional) Frozen Qwen2.5-1.5B-Instruct zero-shot baseline

Skip this cell if you're tight on GPU. The README's results table is fine
with just Random + Heuristic as the published baselines; the LLM zero-shot
result is mainly a sanity-check anchor for notebook 02's trained policy.

When you do run it, set `LLM_EPISODES = 10` per scenario to stay under the
Colab T4 wall clock.

In [ ]:
RUN_LLM_BASELINE = False  # flip to True when you're ready
LLM_EPISODES = 10
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

if RUN_LLM_BASELINE:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from cybersec import ActionType, CybersecAction
    from cybersec.baselines import EpisodeResult

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    model.eval()

    SYSTEM_PROMPT = (
        "You are an SRE-grade cyber-defender driving an OpenEnv environment.\n"
        "Reply with exactly one JSON object on one line of the form\n"
        '{"action_type": "...", "target": "..."}.\n'
        "action_type must be one of MONITOR, INVESTIGATE, ISOLATE_ASSET, REVOKE_IDENTITY, BLOCK_EGRESS, PATCH_ASSET.\n"
        "target must be omitted for MONITOR; otherwise it must come from valid_targets."
    )

    def llm_act(obs):
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": render_observation(obs)},
        ]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=48, do_sample=False, pad_token_id=tok.eos_token_id)
        completion = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        try:
            line = completion.splitlines()[0]
            payload = json.loads(line)
            return CybersecAction(**payload)
        except Exception:
            return CybersecAction(action_type=ActionType.MONITOR)

    class _LLMPolicy:
        name = "qwen-1.5b-zeroshot"
        def reset(self):
            pass
        def act(self, obs):
            return llm_act(obs)

    llm_results = {}
    for scenario_id in SCENARIOS:
        runs = [run_episode(env, _LLMPolicy(), seed=s, scenario_id=scenario_id) for s in range(LLM_EPISODES)]
        llm_results[scenario_id] = aggregate_results(runs)
        print(scenario_id, "->", llm_results[scenario_id])
    metrics["_llm_zeroshot"] = llm_results
    (ARTIFACTS / "baseline_metrics.json").write_text(json.dumps(metrics, indent=2))

## 6. Sanity check — heuristic must beat random on average

If this assertion ever fails on the published reward weights, the reward
model has regressed and the trained policy will be impossible to interpret.

In [ ]:
wins = 0
for scenario_id, cell in results.items():
    r_mean = float(np.mean([r.cumulative_reward for r in cell['random']]))
    h_mean = float(np.mean([r.cumulative_reward for r in cell['heuristic']]))
    print(f"{scenario_id:<32s} random={r_mean:7.3f}  heuristic={h_mean:7.3f}  delta={h_mean - r_mean:+.3f}")
    wins += int(h_mean > r_mean)
print(f"heuristic outperforms random on {wins}/{len(results)} scenarios")
assert wins >= 2, (
    "reward shaping has regressed: the heuristic should beat random on at least 2/3 scenarios"
)